# Project 1: Finding similar book reviews

## Libraries and Dataset Download

In [39]:
!pip install --quiet kaggle pandas

In [40]:
import os
import zipfile
import pandas as pd
import html
import re
from random import seed, randint, sample

In [41]:
os.environ['KAGGLE_USERNAME'] = "..."
os.environ['KAGGLE_KEY'] = "..."
!kaggle datasets download -d mohamedbakhet/amazon-books-reviews

Dataset URL: https://www.kaggle.com/datasets/mohamedbakhet/amazon-books-reviews
License(s): CC0-1.0
amazon-books-reviews.zip: Skipping, found more recently modified local copy (use --force to force download)


In [42]:
with zipfile.ZipFile("amazon-books-reviews.zip", 'r') as archive:
    archive.extractall("data")  

In [43]:
rows = sum(1 for row in open("data/Books_rating.csv", encoding="utf-8")) - 1
print("Total rows:", rows)

Total rows: 3000000


## Dataset Exploration

### Dataset Sampling

In [44]:
path = "data/Books_rating.csv"

full_dataset = False

if full_dataset:
    df = pd.read_csv(path)
else:
    df = pd.read_csv(path, nrows=20000)

print(df.shape)
print(df.head())

(20000, 10)
           Id                           Title  Price         User_id  \
0  1882931173  Its Only Art If Its Well Hung!    NaN   AVCGYZL8FQQTD   
1  0826414346        Dr. Seuss: American Icon    NaN  A30TK6U7DNS82R   
2  0826414346        Dr. Seuss: American Icon    NaN  A3UH4UZ4RSVO82   
3  0826414346        Dr. Seuss: American Icon    NaN  A2MVUWT453QH61   
4  0826414346        Dr. Seuss: American Icon    NaN  A22X4XUPKF66MR   

                          profileName review/helpfulness  review/score  \
0               Jim of Oz "jim-of-oz"                7/7           4.0   
1                       Kevin Killian              10/10           5.0   
2                        John Granger              10/11           5.0   
3  Roy E. Perry "amateur philosopher"                7/7           4.0   
4     D. H. Richards "ninthwavestore"                3/3           4.0   

   review/time                                   review/summary  \
0    940636800           Nice collection of

### Column Selection and Missing Values

In [45]:
df = df[['Id', 'Title', 'review/text']]

print(df.shape)
print(df.head())

(20000, 3)
           Id                           Title  \
0  1882931173  Its Only Art If Its Well Hung!   
1  0826414346        Dr. Seuss: American Icon   
2  0826414346        Dr. Seuss: American Icon   
3  0826414346        Dr. Seuss: American Icon   
4  0826414346        Dr. Seuss: American Icon   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [46]:
# Missing values
missing_values = df[['Id', 'review/text']].isna().sum()
print(missing_values)

Id             0
review/text    0
dtype: int64


### HTML Entities Removal

In [47]:
# Clean html entities
df['review/text'] = df['review/text'].apply(html.unescape)

### Text Processing

In [48]:
# 1. Lowercase 
def to_lowercase(text):
    return text.lower()

# 2. Remove punctuation
def remove_punctuation(text):
    return re.sub(r"[^a-z0-9\s']", ' ', text)

# 3. Tokenization
def tokenize(text):
    return text.split()

# 4. Stopwords removal 
stopwords = {
    "all", "just", "being", "over", "both", "through", "yourselves", "its",
    "before", "herself", "had", "should", "to", "ours", "has", "do", "them",
    "his", "they", "during", "now", "him", "did", "this", "she", "each",
    "further", "where", "few", "because", "doing", "some", "are", "our",
    "ourselves", "what", "for", "while", "does", "above", "between", "be",
    "we", "who", "were", "here", "hers", "by", "on", "about", "of", "against",
    "or", "own", "into", "yourself", "down", "your", "from", "her", "their",
    "there", "been", "whom", "too", "themselves", "was", "until", "more",
    "himself", "that", "but", "with", "than", "those", "he", "me", "myself",
    "these", "up", "will", "theirs", "my", "and", "then", "is", "am", "it",
    "an", "as", "itself", "at", "have", "in", "any", "if", "again", "when",
    "same", "how", "other", "which", "you", "after", "most", "such", "why",
    "a", "off", "i", "yours", "so", "the", "having", "once"
}

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stopwords]


In [49]:
def preprocess(text):
    text = to_lowercase(text)
    text = remove_punctuation(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    return tokens

df['tokens'] = df['review/text'].apply(preprocess)

## Shingles generation

In [50]:
# Shingles
K_SHINGLE = 2

def to_shingles(tokens, k):
    tokens = list(tokens)
    if len(tokens) < k:
        return set()
    return set(" ".join(tokens[i:i+k]) for i in range(len(tokens)-k+1))

df["features"] = df["tokens"].map(lambda t: to_shingles(t, K_SHINGLE))

df = df.reset_index(drop=True)

print("Review example:")
print(df.loc[0, "review/text"])
print("\nShingles examples:", list(df.loc[0, "features"])[:10])

Review example:
This is only for Julie Strain fans. It's a collection of her photos -- about 80 pages worth with a nice section of paintings by Olivia.If you're looking for heavy literary content, this isn't the place to find it -- there's only about 2 pages with text and everything else is photos.Bottom line: if you only want one book, the Six Foot One ... is probably a better choice, however, if you like Julie like I like Julie, you won't go wrong on this one either.

Shingles examples: ["it's collection", 'julie strain', 'text everything', "there's only", "content isn't", 'book six', "find there's", 'pages worth', 'line only', 'only 2']


## Similarity Detection

### MinHash Signatures

In [51]:
num_hashes = 100
prime = 4294967311
num_bands = 25
rows_per_band = num_hashes // num_bands

seed(42)
hash_params = [(randint(1, prime - 1), randint(0, prime - 1)) for _ in range(num_hashes)]


def minhash_signature_filtered(shingles):
    if not shingles:
        return None
    signature = []
    for a, b in hash_params:
        min_hash = min(((a * hash(shingle) + b) % prime) for shingle in shingles)
        signature.append(min_hash)
    return signature

df['minhash_signature'] = df['features'].apply(minhash_signature_filtered)

df_valid = df[df['minhash_signature'].notnull()].reset_index(drop=True)

### Locality-Sensitive Hashing (LSH)

In [52]:
def lsh_buckets(signature):
    buckets = []
    for i in range(num_bands):
        start = i * rows_per_band
        end = (i + 1) * rows_per_band
        band = tuple(signature[start:end])
        buckets.append(hash(band))
    return buckets

buckets = {}
for idx, signature in enumerate(df_valid['minhash_signature']):
    for bucket in lsh_buckets(signature):
        if bucket not in buckets:
            buckets[bucket] = []
        buckets[bucket].append(idx)


candidate_pairs = set()
for idx_list in buckets.values():
    if len(idx_list) > 1:
        for i in range(len(idx_list)):
            for j in range(i + 1, len(idx_list)):
                candidate_pairs.add((idx_list[i], idx_list[j]))

print(f"Candidate pairs to compare: {len(candidate_pairs)}")

Candidate pairs to compare: 1327


### Jaccard similarity and results

In [65]:
def approx_jaccard(sig1, sig2):
    return sum(1 for k in range(num_hashes) if sig1[k] == sig2[k]) / num_hashes

results = []
for i, j in candidate_pairs:
    similarity = approx_jaccard(df_valid.loc[i, 'minhash_signature'], df_valid.loc[j, 'minhash_signature'])
    results.append((i, j, similarity))

threshold_min = 0.5
threshold_max = 0.95
similar_pairs = []
for i, j, similarity in results:
    if threshold_min <= similarity <= threshold_max:
        similar_pairs.append((i, j, similarity))

print(f"Pairs with similarity between {threshold_min} and {threshold_max}: {len(similar_pairs)}")
print(similar_pairs[:10])

for idx, (i, j, similarity) in enumerate(similar_pairs):
    print(f"Pair {idx+1}: Similarity {similarity:.3f}")
    print(f"Index i: {i}")
    print(f"Title i: {df_valid.loc[i, 'Title']}")
    print(f"Review i: {df_valid.loc[i, 'review/text']}")
    print(f"Index j: {j}")
    print(f"Title j: {df_valid.loc[j, 'Title']}")
    print(f"Review j: {df_valid.loc[j, 'review/text']}")
    print("="*80)

Pairs with similarity between 0.5 and 0.95: 111
[(9008, 9026, 0.84), (8000, 10088, 0.95), (7601, 7760, 0.95), (9816, 9817, 0.84), (3933, 3945, 0.61), (9009, 9161, 0.93), (15318, 15788, 0.9), (2457, 11606, 0.73), (7152, 7188, 0.91), (15005, 15835, 0.95)]
Pair 1: Similarity 0.840
Index i: 9008
Title i: Romeo and Juliet
Review i: Two teenagers from rival families fall in love, marry secretly, and take their own lives rather than live without each other. Despite the teenage melodrama, "Romeo and Juliet" remains one of Shakespeare's most enduring and popular plays, even if it wasn't his best -- lots of death, teen lovers and enchanting dialogue.In the city of Verona, the Montagues and Capulets are locked in a deadly feud. Then a Montague teen named Romeo, infatuated with a Capulet girl named Rosaline, sneaks into a party to see her.... but instead encounters another Capulet girl named Juliet, and the two immediately fall in love. Since their families hate each other, their love must be expr